In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
import random
import re

from sklearn.feature_extraction.text import TfidfVectorizer

from typing import Dict,List, Optional, Tuple
from dataclasses import dataclass

from sentence_transformers import SentenceTransformer

In [2]:
def set_seed(seed = 42):
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

Предметная область: Git (ветки, коммиты, PR, репозитории)

In [3]:
documents = [
    {"doc_id": "doc_0", "title": "Создание новой ветки", "text": "Инструкция: Как создать новую ветку в Git: git branch <branch-name>"},
    {"doc_id": "doc_1", "title": "Переключение между ветками", "text": "Инструкция: Как переключиться между ветками: git checkout <branch-name>"},
    {"doc_id": "doc_2", "title": "Создание ветки и переключение на нее", "text": "Создание ветки и переключение на нее: Как создать ветку и сразу переключиться на неё: git checkout -b <branch-name>"},
    {"doc_id": "doc_3", "title": "Удаление ветки", "text": "Инструкция: Как удалить ветку локально: git branch -d <branch-name>"},
    {"doc_id": "doc_4", "title": "Что такое pull request", "text": "FAQ: Что такое pull request? Это запрос на включение изменений из одной ветки в другую в удалённом репозитории."},
    {"doc_id": "doc_5", "title": "Чем отличается git pull от git fetch", "text": "FAQ: Чем отличается git pull от git fetch? git fetch только загружает изменения, а git pull загружает и сразу сливает с текущей веткой."},
    {"doc_id": "doc_6", "title": "Что такое merge conflict", "text": "FAQ: Что такое merge conflict? Это ситуация, когда Git не может автоматически объединить изменения из разных веток из-за конфликтующих правок в одних и тех же строках файла."},
    {"doc_id": "doc_7", "title": "Сколько проводится код-ревью", "text": "Регламент: Код-ревью должно проводиться не более 2 часов после создания PR."},
    {"doc_id": "doc_8", "title": "Формат названия веток", "text": "Регламент: Названия веток должны соответствовать формату: feature/описание, bugfix/описание или hotfix/описание."},
    {"doc_id": "doc_9", "title": "Что сделать перед PR", "text": "Регламент: Перед созданием PR необходимо убедиться, что все тесты проходят успешно."},
    {"doc_id": "doc_10", "title": "Правила пуша", "text": "Регламент: Запрещено пушить непосредственно в ветку main — только через pull request с ревью минимум одного разработчика."},
    {"doc_id": "doc_11", "title": "Отправка в удаленный репозиторий", "text": "Инструкция: Как отправить изменения в удалённый репозиторий: git push origin <branch-name>"},
    {"doc_id": "doc_12", "title": "Загрузка из удаленного репозитория", "text": "Инструкция: Как загрузить изменения из удалённого репозитория: git pull origin <branch-name>"},
]

In [4]:
benchmark_queries = [
    {
        "query_id": "q01",
        "query": "Что нужно сделать перед созданием PR?",
        "relevant_doc_ids": ["9"]
    },
    {
        "query_id": "q02",
        "query": "Что такое merge conflict?",
        "relevant_doc_ids": ["6"]
    },
    {
        "query_id": "q03",
        "query": "Как переключиться между ветками?",
        "relevant_doc_ids": ["1"]
    },
    {
        "query_id": "q04",
        "query": "Сколько длится код-ревью?",
        "relevant_doc_ids": ["7"]
    },
        {
        "query_id": "q05",
        "query": "Что такое pull request?",
        "relevant_doc_ids": ["4"]
    },
    {
        "query_id": "q06",
        "query": "Как отправить измененные данные в удаленный репозиторий?",
        "relevant_doc_ids": ["11"]
    },
    {
        "query_id": "q07",
        "query": "Чем отличается git pull от git fetch?",
        "relevant_doc_ids": ["5"]
    },
    {
        "query_id": "q08",
        "query": "Как удалить ветку?",
        "relevant_doc_ids": ["3"]
    }
]

benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

,query_id,query,relevant_doc_ids
0,q01,Что нужно сделать перед созданием PR?,[9]
1,q02,Что такое merge conflict?,[6]
2,q03,Как переключиться между ветками?,[1]
3,q04,Сколько длится код-ревью?,[7]
4,q05,Что такое pull request?,[4]
5,q06,Как отправить измененные данные в удаленный ре...,[11]
6,q07,Чем отличается git pull от git fetch?,[5]
7,q08,Как удалить ветку?,[3]


Предметная область Git (ветки, коммиты, PR, репозитории) представляет собой базу знаний по работе и использованию Git, а именно какие комманды и флаги использовать, советы по использованию PR и т.д. 

Эта база знаний хорошо подходит для retrieval / mini-RAG задачи, так как было бы очень удобно находить готовый ответ на интересующий вопрос.

In [5]:
def chunking(text, chunk_size= 10, overlap = 3):
    words= text.split()
    chunks =[]

    step = chunk_size-overlap

    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]

        chunks.append(" ".join(chunk_words))
        
        if end > len(words):
            break

    return chunks

In [6]:
for i in range(len(documents)):
    print( chunking(documents[i]["text"]))

['Инструкция: Как создать новую ветку в Git: git branch <branch-name>', 'git branch <branch-name>']
['Инструкция: Как переключиться между ветками: git checkout <branch-name>']
['Создание ветки и переключение на нее: Как создать ветку и', 'создать ветку и сразу переключиться на неё: git checkout -b', 'git checkout -b <branch-name>']
['Инструкция: Как удалить ветку локально: git branch -d <branch-name>']
['FAQ: Что такое pull request? Это запрос на включение изменений', 'на включение изменений из одной ветки в другую в удалённом', 'другую в удалённом репозитории.']
['FAQ: Чем отличается git pull от git fetch? git fetch', 'fetch? git fetch только загружает изменения, а git pull загружает', 'git pull загружает и сразу сливает с текущей веткой.']
['FAQ: Что такое merge conflict? Это ситуация, когда Git не', 'когда Git не может автоматически объединить изменения из разных веток', 'из разных веток из-за конфликтующих правок в одних и тех', 'одних и тех же строках файла.']
['Регламент: Код-рев

Я выбрала именно такие chunk_size и overlap потому что они оптимально разбивают текст с небольшым количесвом повторов, что будет удобно для последующей работы с текстом.

In [7]:
def build_chunks_df(docs, chunk_size = 10, overlap = 5):
    rows = []

    for i in range(len(docs)):
        chunks =  chunking(docs[i]["text"])
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": i,
                    "title": docs[i]["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split())
                    
                }
            )
    return pd.DataFrame(rows)

chunks_df  = build_chunks_df(documents)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))

Количество чанков: 30


,doc_id,title,chunk_id,chunk_text,n_words
0,0,Создание новой ветки,0,Инструкция: Как создать новую ветку в Git: git...,10
1,0,Создание новой ветки,1,git branch <branch-name>,3
2,1,Переключение между ветками,0,Инструкция: Как переключиться между ветками: g...,8
3,2,Создание ветки и переключение на нее,0,Создание ветки и переключение на нее: Как созд...,10
4,2,Создание ветки и переключение на нее,1,создать ветку и сразу переключиться на неё: gi...,10
5,2,Создание ветки и переключение на нее,2,git checkout -b <branch-name>,4
6,3,Удаление ветки,0,Инструкция: Как удалить ветку локально: git br...,9
7,4,Что такое pull request,0,FAQ: Что такое pull request? Это запрос на вкл...,10
8,4,Что такое pull request,1,на включение изменений из одной ветки в другую...,10
9,4,Что такое pull request,2,другую в удалённом репозитории.,4


In [8]:
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=6,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=6,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

In [9]:
@dataclass
class RetrievalArtifacts:
    backend_name: str
    backend: EmbeddingBackend
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    index: object

In [10]:
def build_embedding(model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device: str = "cpu"):
    backend = SentenceTransformersBackend(model_name=model_name, device = device)

    return backend

In [11]:
embedder = build_embedding()

In [12]:
chunking_text = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunking_text)

In [13]:
print("Форма матрицы эмбеддингов:", chunk_embeddings.shape)

# Проверяем длины векторов.
# Если normalize_embeddings=True сработал корректно, все нормы должны быть ≈ 1.0.
# Это означает, что косинусное сходство далее можно считать через скалярное произведение.
vector_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Минимальная норма:", round(float(vector_norms.min()), 4))
print("Максимальная норма:", round(float(vector_norms.max()), 4))
print("Средняя норма:", round(float(vector_norms.mean()), 4))
print("→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.")

Форма матрицы эмбеддингов: (30, 384)
Минимальная норма: 1.0
Максимальная норма: 1.0
Средняя норма: 1.0
→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.


Построение faiss

In [14]:
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None

        self._faiss_index = faiss.IndexFlatIP(dim)
        self.backend_name = "FAISS IndexFlatIP"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices



In [15]:
search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

In [16]:
def search_similar_chunks(query, top_k):
    query_vector = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vector)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start = 1):
        chunk_row = chunks_df.iloc[int(idx)]

        rows.append({
            "rank": rank,
            "doc_id": chunk_row["doc_id"],
            "title": chunk_row["title"],
            "chunk_id": int(chunk_row["chunk_id"]),
            "score": round(float(score), 4),
            "chunk_text": chunk_row["chunk_text"]
        })

    return pd.DataFrame(rows)

In [17]:
faiss_query = ["Что нужно сделать перед созданием PR?",
                "Как создать новую ветку в Git?",
                "Как удалить ветку?",
                "Какой формат у веток?"]

for i in faiss_query:
    print(f"Вопрос: {i}")
    faiss_results_df = search_similar_chunks(i, top_k = 3)
    display(faiss_results_df)
    print("\n")

display(faiss_results_df)

Вопрос: Что нужно сделать перед созданием PR?


,rank,doc_id,title,chunk_id,score,chunk_text
0,1,7,Сколько проводится код-ревью,1,0.6880,часов после создания PR.
1,2,9,Что сделать перед PR,0,0.4879,Регламент: Перед созданием PR необходимо убеди...
2,3,7,Сколько проводится код-ревью,0,0.2871,Регламент: Код-ревью должно проводиться не бол...
3,4,10,Правила пуша,2,0.2765,минимум одного разработчика.
4,5,8,Формат названия веток,1,0.2335,bugfix/описание или hotfix/описание.




Вопрос: Как создать новую ветку в Git?


,rank,doc_id,title,chunk_id,score,chunk_text
0,1,0,Создание новой ветки,0,0.8699,Инструкция: Как создать новую ветку в Git: git...
1,2,6,Что такое merge conflict,1,0.7570,когда Git не может автоматически объединить из...
2,3,2,Создание ветки и переключение на нее,1,0.6892,создать ветку и сразу переключиться на неё: gi...
3,4,5,Чем отличается git pull от git fetch,2,0.6725,git pull загружает и сразу сливает с текущей в...
4,5,11,Отправка в удаленный репозиторий,0,0.6594,Инструкция: Как отправить изменения в удалённы...




Вопрос: Как удалить ветку?


,rank,doc_id,title,chunk_id,score,chunk_text
0,1,3,Удаление ветки,0,0.5332,Инструкция: Как удалить ветку локально: git br...
1,2,2,Создание ветки и переключение на нее,0,0.5330,Создание ветки и переключение на нее: Как созд...
2,3,4,Что такое pull request,1,0.4836,на включение изменений из одной ветки в другую...
3,4,6,Что такое merge conflict,2,0.3295,из разных веток из-за конфликтующих правок в о...
4,5,5,Чем отличается git pull от git fetch,2,0.2878,git pull загружает и сразу сливает с текущей в...




Вопрос: Какой формат у веток?


,rank,doc_id,title,chunk_id,score,chunk_text
0,1,8,Формат названия веток,0,0.4582,Регламент: Названия веток должны соответствова...
1,2,6,Что такое merge conflict,3,0.4056,одних и тех же строках файла.
2,3,6,Что такое merge conflict,2,0.3910,из разных веток из-за конфликтующих правок в о...
3,4,8,Формат названия веток,1,0.3873,bugfix/описание или hotfix/описание.
4,5,4,Что такое pull request,2,0.3623,другую в удалённом репозитории.


,rank,doc_id,title,chunk_id,score,chunk_text
0,1,8,Формат названия веток,0,0.4582,Регламент: Названия веток должны соответствова...
1,2,6,Что такое merge conflict,3,0.4056,одних и тех же строках файла.
2,3,6,Что такое merge conflict,2,0.3910,из разных веток из-за конфликтующих правок в о...
3,4,8,Формат названия веток,1,0.3873,bugfix/описание или hotfix/описание.
4,5,4,Что такое pull request,2,0.3623,другую в удалённом репозитории.


In [18]:
benchmark_queries = [
    {
        "query_id": "q01",
        "query": "Что нужно сделать перед созданием PR?",
        "relevant_doc_ids": ["9"]
    },
    {
        "query_id": "q02",
        "query": "Что такое merge conflict?",
        "relevant_doc_ids": ["6"]
    },
    {
        "query_id": "q03",
        "query": "Как переключиться между ветками?",
        "relevant_doc_ids": ["2"]
    },
    {
        "query_id": "q04",
        "query": "Сколько длится код-ревью?",
        "relevant_doc_ids": ["7"]
    },
        {
        "query_id": "q05",
        "query": "Что такое pull request?",
        "relevant_doc_ids": ["4"]
    },
    {
        "query_id": "q06",
        "query": "Как отправить измененные данные в удаленный репозиторий?",
        "relevant_doc_ids": ["11"]
    },
    {
        "query_id": "q07",
        "query": "Чем отличается git pull от git fetch?",
        "relevant_doc_ids": ["5"]
    },
    {
        "query_id": "q08",
        "query": "Как удалить ветку?",
        "relevant_doc_ids": ["3"]
    }
]

benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

,query_id,query,relevant_doc_ids
0,q01,Что нужно сделать перед созданием PR?,[9]
1,q02,Что такое merge conflict?,[6]
2,q03,Как переключиться между ветками?,[2]
3,q04,Сколько длится код-ревью?,[7]
4,q05,Что такое pull request?,[4]
5,q06,Как отправить измененные данные в удаленный ре...,[11]
6,q07,Чем отличается git pull от git fetch?,[5]
7,q08,Как удалить ветку?,[3]


In [19]:
def build_chunks(
    docs: List[Dict[str, str]],
    chunk_size: int,
    overlap: int,
):

    rows: List[Dict[str, object]] = []
    for idx, doc in enumerate(docs):
        doc_id = doc.get("doc_id", f"doc_{idx}")
        title = doc.get("title", doc_id)
        parts = chunking(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_idx, chunk_text_value in enumerate(parts):
            rows.append(
                {
                    "doc_id": doc_id,
                    "title": title,
                    "chunk_id": f"{doc_id}_chunk_{chunk_idx}",
                    "chunk_idx": chunk_idx,
                    "chunk_text": chunk_text_value,
                }
            )
    return rows

In [20]:
def select_backend(device: str = "cpu"):
        backend = SentenceTransformersBackend(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            device=device,
        )
        return backend

In [21]:
def build_retriever(
    docs: List[Dict[str, str]],
    chunk_size: int =10,
    overlap: int =5,
    device: str = "cpu",
):
    chunks = build_chunks(docs, chunk_size=chunk_size, overlap=overlap)
    chunks_df = pd.DataFrame(chunks)

    backend = select_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())

    index = faiss.IndexFlatIP(chunk_vectors.shape[1])
    index.add(chunk_vectors)

    return RetrievalArtifacts(
        backend_name=backend.backend_name,
        backend=backend,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        index=index,
    )

In [22]:
def search_chunks(
    query: str,
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
):
    query_vector = artifacts.backend.encode_queries([query]).astype("float32")
    scores, indices = artifacts.index.search(query_vector, top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = artifacts.chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": chunk_row["chunk_id"],
                "chunk_text": chunk_row["chunk_text"],
            }
        )
    return pd.DataFrame(rows)

In [23]:
def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered


def evaluate_query(
    query: str,
    relevant_doc_ids: List[str],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts=artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }

In [ ]:
def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            artifacts=artifacts,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "expected_source": ", ".join(row["relevant_doc_ids"]),
                "retrieved_sources": ", ".join(metrics["predicted_doc_ids"]),
                f"hit@{top_k}": metrics["hit"],
                f"recall@{top_k}": metrics["recall"],
                f"MRR@{top_k}": metrics["mrr"],
                "hit_at_k": metrics["first_relevant_rank"],
            }
        )


    return pd.DataFrame(rows)

In [73]:
artifacts = build_retriever(
    docs=[{"text": doc["text"],"title": f"doc_{i}", "doc_id": str(i)} for i, doc in enumerate(documents)],
    chunk_size=10,
    overlap=5,
    device="cpu"
)

eval_results = evaluate_benchmark(
    benchmark_rows=benchmark_queries,
    artifacts=artifacts,
    top_k=3
)

display(eval_results)

,query_id,query,expected_source,retrieved_sources,hit@3,recall@3,MRR@3,hit_at_k
0,q01,Что нужно сделать перед созданием PR?,9,"7, 9",1,1.0,0.5,2
1,q02,Что такое merge conflict?,6,6,1,1.0,1.0,1
2,q03,Как переключиться между ветками?,2,"2, 5, 4",1,1.0,1.0,1
3,q04,Сколько длится код-ревью?,7,"7, 2",1,1.0,1.0,1
4,q05,Что такое pull request?,4,"4, 10, 5",1,1.0,1.0,1
5,q06,Как отправить измененные данные в удаленный ре...,11,"11, 12, 4",1,1.0,1.0,1
6,q07,Чем отличается git pull от git fetch?,5,5,1,1.0,1.0,1
7,q08,Как удалить ветку?,3,"3, 2, 5",1,1.0,1.0,1


In [58]:
chunk_configs = [
    {"chunk_size": 6, "overlap": 4},
    {"chunk_size": 13, "overlap": 5},
    {"chunk_size": 20, "overlap": 7},
    {"chunk_size": 23, "overlap": 10},
]

chunk_experiments = []

for cfg in chunk_configs:
    docs_for_retriever = [
        {
            "text": doc["text"],
            "title": doc["title"],
            "doc_id": str(i)
        }
        for i, doc in enumerate(documents)
    ]
    
    exp_artifacts = build_retriever(
        docs=docs_for_retriever,
        chunk_size=cfg["chunk_size"],
        overlap=cfg["overlap"],
        device="cpu"
    )
    
    eval_df = evaluate_benchmark(
        benchmark_rows=benchmark_queries,
        artifacts=exp_artifacts,
        top_k=3
    )

    chunk_experiments.append(
        {
            "chunk_size": cfg["chunk_size"],
            "overlap": cfg["overlap"],
            "num_chunks": len(exp_artifacts.chunks_df),
            "backend_name": exp_artifacts.backend_name,
            "mean_hit@3": eval_df["hit@3"].mean(),
            "mean_recall@3": eval_df["recall@3"].mean(),
            "mean_MRR@3": eval_df["MRR@3"].mean(),
        }
    )

chunk_experiments_df = pd.DataFrame(chunk_experiments).sort_values(
    by=["mean_hit@3", "mean_MRR@3", "num_chunks"],
    ascending=[False, False, True],
).reset_index(drop=True)

results_df = pd.DataFrame(chunk_experiments)
display(results_df)

,chunk_size,overlap,num_chunks,backend_name,mean_hit@3,mean_recall@3,mean_MRR@3
0,6,4,75,SentenceTransformer: sentence-transformers/par...,1.0,1.0,1.0000
1,13,5,20,SentenceTransformer: sentence-transformers/par...,1.0,1.0,0.9375
2,20,7,15,SentenceTransformer: sentence-transformers/par...,1.0,1.0,0.9375
3,23,10,15,SentenceTransformer: sentence-transformers/par...,1.0,1.0,0.9375


In [59]:
new_documents = [
    {
        "doc_id": "doc_13",
        "title": "Проверка статуса репозитория",
        "text": "Просмотреть статус нужного репозитория можно по ключевому слову status: его действие распространяется на подготовленные, неподготовленные и неотслеживаемые файлы.",
    },
    {
        "doc_id": "doc_14",
        "title": "Просмотр изменений до коммита",
        "text": "Можно просматривать список изменений, внесённых в репозиторий, используя параметр diff. По умолчанию отображаются только изменения, не подготовленные для фиксации.",
    },
    {
        "doc_id": "doc_15",
        "title": "Откат последнего коммита",
        "text": "Откатить последний коммит можно с помощью параметра revert. Создастся новый коммит, содержащий обратные преобразования относительно предыдущего, и добавится к истории текущей ветки.",
    },
]

updated_documents = documents + new_documents

In [60]:
new_queries = [
    "Как откатить последний коммит?",
    "Как посмотреть статус репозитория?",
]

for query in new_queries:
    display(f"**Запрос:** {query}")
    display(search_chunks(query, artifacts=artifacts, top_k=3)[["rank", "score", "doc_id", "title", "chunk_text"]])

'**Запрос:** Как откатить последний коммит?'

,rank,score,doc_id,title,chunk_text
0,1,0.337693,7,doc_7,более 2 часов после создания PR.
1,2,0.297134,4,doc_4,из одной ветки в другую в удалённом репозитории.
2,3,0.296197,10,doc_10,ветку main — только через pull request с ревью...


'**Запрос:** Как посмотреть статус репозитория?'

,rank,score,doc_id,title,chunk_text
0,1,0.425138,9,doc_9,Регламент: Перед созданием PR необходимо убеди...
1,2,0.381290,9,doc_9,"убедиться, что все тесты проходят успешно."
2,3,0.351550,4,doc_4,из одной ветки в другую в удалённом репозитории.


In [61]:
updated_artifacts = build_retriever(
    docs=[{"text": doc["text"], "title": doc["title"], "doc_id": str(i)} for i, doc in enumerate(updated_documents)],
    chunk_size=10,
    overlap=4,
)


extended_benchmark_queries = benchmark_queries + [
    {
        "query_id": "q09",
        "query": "Как откатить последний коммит?",
        "relevant_doc_ids": ["15"],
    },
    {
        "query_id": "q10",
        "query": "Как посмотреть статус репозитория?",
        "relevant_doc_ids": ["13"],
    },
]


In [62]:
before_update_eval = evaluate_benchmark(extended_benchmark_queries, artifacts=artifacts, top_k=3)
after_update_eval = evaluate_benchmark(extended_benchmark_queries, artifacts=updated_artifacts, top_k=3)

In [ ]:
def compare_retrieval(row):
    before = set(row["before_retrieved_sources"].split(", ")) if pd.notna(row["before_retrieved_sources"]) else set()
    after = set(row["after_retrieved_sources"].split(", ")) if pd.notna(row["after_retrieved_sources"]) else set()
    
    row["changed"] = ", ".join(after - before) if (after - before) else "none"
    
    return row

In [ ]:
comparison_df = before_update_eval.merge(
    after_update_eval,
    on=["query_id", "query", "relevant_doc_ids"],
    suffixes=("_before", "_after"),
)

display(comparison_df)

summary_comparison_df = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "before_update": [
            before_update_eval["hit@3"].mean(),
            before_update_eval["recall@3"].mean(),
            before_update_eval["MRR@3"].mean(),
        ],
        "after_update": [
            after_update_eval["hit@3"].mean(),
            after_update_eval["recall@3"].mean(),
            after_update_eval["MRR@3"].mean(),
        ],
    }
)
summary_comparison_df["delta"] = summary_comparison_df["after_update"] - summary_comparison_df["before_update"]

comparison_df.rename(columns = {"expected_source_before":"before_retrieved_sources"}, inplace = True)
comparison_df.rename(columns = {"expected_source_after":"after_retrieved_sources"}, inplace = True)

comparison_df = comparison_df.apply(compare_retrieval, axis=1)
comparison_df.to_csv("artifacts/retrieval_before_after_update.csv")

,query_id,query,relevant_doc_ids,expected_source_before,hit@3_before,recall@3_before,MRR@3_before,hit_at_k_before,expected_source_after,hit@3_after,recall@3_after,MRR@3_after,hit_at_k_after
0,q01,Что нужно сделать перед созданием PR?,9,"7, 9",1,1.0,0.5,2.0,"7, 9, 13",1,1.0,0.5,2
1,q02,Что такое merge conflict?,6,6,1,1.0,1.0,1.0,6,1,1.0,1.0,1
2,q03,Как переключиться между ветками?,2,"2, 5, 4",1,1.0,1.0,1.0,"2, 6, 4",1,1.0,1.0,1
3,q04,Сколько длится код-ревью?,7,"7, 2",1,1.0,1.0,1.0,"7, 2, 5",1,1.0,1.0,1
4,q05,Что такое pull request?,4,"4, 10, 5",1,1.0,1.0,1.0,"4, 10, 5",1,1.0,1.0,1
5,q06,Как отправить измененные данные в удаленный ре...,11,"11, 12, 4",1,1.0,1.0,1.0,"11, 14, 13",1,1.0,1.0,1
6,q07,Чем отличается git pull от git fetch?,5,5,1,1.0,1.0,1.0,5,1,1.0,1.0,1
7,q08,Как удалить ветку?,3,"3, 2, 5",1,1.0,1.0,1.0,"3, 2, 5",1,1.0,1.0,1
8,q09,Как откатить последний коммит?,15,"7, 4, 10",0,0.0,0.0,NaN,15,1,1.0,1.0,1
9,q10,Как посмотреть статус репозитория?,13,"9, 4",0,0.0,0.0,NaN,"13, 15",1,1.0,1.0,1


,query_id,query,relevant_doc_ids,before_retrieved_sources,hit@3_before,recall@3_before,MRR@3_before,hit_at_k_before,after_retrieved_sources,hit@3_after,recall@3_after,MRR@3_after,hit_at_k_after,changed
0,q01,Что нужно сделать перед созданием PR?,9,"7, 9",1,1.0,0.5,2.0,"7, 9, 13",1,1.0,0.5,2,13
1,q02,Что такое merge conflict?,6,6,1,1.0,1.0,1.0,6,1,1.0,1.0,1,none
2,q03,Как переключиться между ветками?,2,"2, 5, 4",1,1.0,1.0,1.0,"2, 6, 4",1,1.0,1.0,1,6
3,q04,Сколько длится код-ревью?,7,"7, 2",1,1.0,1.0,1.0,"7, 2, 5",1,1.0,1.0,1,5
4,q05,Что такое pull request?,4,"4, 10, 5",1,1.0,1.0,1.0,"4, 10, 5",1,1.0,1.0,1,none
5,q06,Как отправить измененные данные в удаленный ре...,11,"11, 12, 4",1,1.0,1.0,1.0,"11, 14, 13",1,1.0,1.0,1,"13, 14"
6,q07,Чем отличается git pull от git fetch?,5,5,1,1.0,1.0,1.0,5,1,1.0,1.0,1,none
7,q08,Как удалить ветку?,3,"3, 2, 5",1,1.0,1.0,1.0,"3, 2, 5",1,1.0,1.0,1,none
8,q09,Как откатить последний коммит?,15,"7, 4, 10",0,0.0,0.0,NaN,15,1,1.0,1.0,1,15
9,q10,Как посмотреть статус репозитория?,13,"9, 4",0,0.0,0.0,NaN,"13, 15",1,1.0,1.0,1,"15, 13"


Mini-RAG

In [64]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

In [65]:
def build_context_from_retrieval(query: str, artifacts: RetrievalArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

In [66]:
query = "Как посмотреть изменения внесенные в коммит?"
context, retrieved_df = build_context_from_retrieval(query, artifacts=artifacts, top_k=3)

display(f"### Запрос: {query}")
display(retrieved_df)
print(context)

'### Запрос: Как посмотреть изменения внесенные в коммит?'

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.502350,4,doc_4,4_chunk_1,Это запрос на включение изменений из одной вет...
1,2,0.448223,6,doc_6,6_chunk_2,может автоматически объединить изменения из ра...
2,3,0.371679,9,doc_9,9_chunk_0,Регламент: Перед созданием PR необходимо убеди...


[Источник: 4 | doc_4 | score=0.5024]
Это запрос на включение изменений из одной ветки в другую

[Источник: 6 | doc_6 | score=0.4482]
может автоматически объединить изменения из разных веток из-за конфликтующих правок

[Источник: 9 | doc_9 | score=0.3717]
Регламент: Перед созданием PR необходимо убедиться, что все тесты проходят


In [67]:
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)

In [68]:
answer_example = generate_answer_from_context(query, context)
print(answer_example)

может автоматически объединить изменения из разных веток из-за конфликтующих правок


In [69]:
def mini_rag_answer(
    query: str,
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }

In [70]:
last_query = ["Объясни merge conflict?", "Как создать ветку?", "Формат имен веток"]

In [71]:
mini_rag = []
for i in last_query:


    rag_result = mini_rag_answer(
        i,
        artifacts=artifacts,
        top_k=3,
    )

    mini_rag.append({
        "question":rag_result['query'],
        "answer": rag_result['answer'],
        "retrieved_sources": rag_result["sources"]
    })

rag = pd.DataFrame(mini_rag)
rag.to_csv("artifacts/rag_examples.csv")